In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True
)

val_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    '/kaggle/working/dataset/train',
    target_size=(224,224),
    batch_size=16,
    class_mode='binary'
)

val_data = val_gen.flow_from_directory(
    '/kaggle/working/dataset/val',
    target_size=(224,224),
    batch_size=16,
    class_mode='binary'
)

In [ ]:
base_model = MobileNet(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

In [ ]:
for layer in base_model.layers:
    layer.trainable = False

In [ ]:
x = base_model.output
x = GlobalAveragePooling2D()(x)

x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)

output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)

In [ ]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=[early_stop]
)

In [ ]:
model.evaluate(val_data)

In [ ]:
import matplotlib.pyplot as plt

# Accuracy
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')

plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Training vs Validation Accuracy')
plt.legend()

plt.show()

In [ ]:
# Loss
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')

plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training vs Validation Loss')
plt.legend()

plt.show()

In [ ]:
import os

test_dir = "/kaggle/working/dataset/test"

test_images = os.listdir(test_dir)
print(len(test_images))

In [ ]:
import os
import numpy as np
from tensorflow.keras.preprocessing import image

test_dir = "/kaggle/working/dataset/test"

for class_name in os.listdir(test_dir):   # muffin, chihuahua
    class_path = os.path.join(test_dir, class_name)
    
    for img_name in os.listdir(class_path):   # actual images
        img_path = os.path.join(class_path, img_name)
        
        img = image.load_img(img_path, target_size=(224,224))
        img_array = image.img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)
        
        prediction = model.predict(img_array)
        
        print(img_name, "->", prediction)

In [ ]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing import image

test_dir = "/kaggle/working/dataset/test"

data = []

for class_name in os.listdir(test_dir):   # muffin, chihuahua
    class_path = os.path.join(test_dir, class_name)
    
    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)
        
        img = image.load_img(img_path, target_size=(224,224))
        img_array = image.img_to_array(img) / 255.0
        img_array = np.expand_dims(img_array, axis=0)
        
        prediction = model.predict(img_array)[0][0]
        
        # Convert to label
        label = 1 if prediction > 0.5 else 0
        
        data.append([img_name, label])

# Create dataframe
df = pd.DataFrame(data, columns=["image_id", "label"])

# Save CSV
df.to_csv("submission.csv", index=False)

print("CSV file created!")

In [ ]:
import os
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing import image

test_dir = "/kaggle/working/dataset/test"

data = []

for class_name in os.listdir(test_dir):
    class_path = os.path.join(test_dir, class_name)
    
    for img_name in os.listdir(class_path):
        img_path = os.path.join(class_path, img_name)
        
        img = image.load_img(img_path, target_size=(224,224))
        img_array = image.img_to_array(img) / 255.0
        img_array = np.expand_dims(img_array, axis=0)
        
        prediction = model.predict(img_array)[0][0]
        
        # 👇 Convert to class name
        label = "chihuahua" if prediction > 0.5 else "muffin"
        
        data.append([img_name, label])

df = pd.DataFrame(data, columns=["image_id", "label"])
df.to_csv("submission.csv", index=False)

print("CSV with names created!")